# Floor-Level Daily Staffing & Clinical Summary Generation

Aggregates day-shift staffing (08:00–18:00, excluding nursing directors `ladder=5`), patient clinical events, indoor beacon room time, subjective busyness, and care acuity metrics into `floor_daily_staffing_summary.csv`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

sys.path.insert(0, str(Path.cwd().resolve().parent))
from src.config_loader import load_config

cfg = load_config()
PATHS = cfg["resolved_paths"]
ANALYSIS_CFG = cfg["analysis"]
HOURS = list(ANALYSIS_CFG["day_shift_hours"])
LADDER_MAP = {int(k): int(v) for k, v in ANALYSIS_CFG["ladder_to_tier_weight"].items()}
DAY_SHIFT_TYPE = ANALYSIS_CFG["day_shift_type"]

## 1. Data Loading & Shift Filtering

In [ ]:
m_shift = pd.read_csv(PATHS["m_shift"])
m_user = pd.read_csv(PATHS["m_user"])
t_daily_shift = pd.read_csv(PATHS["t_daily_shift"])

day_shift_ids = m_shift[m_shift["shift_type"] == DAY_SHIFT_TYPE]["shift_id"].values

df = (
    t_daily_shift[t_daily_shift["shift_id"].isin(day_shift_ids)]
    .merge(m_shift[["shift_id", "start_time", "end_time"]], on="shift_id", how="left")
    .merge(m_user[["user_id", "ladder", "floor"]], on="user_id", how="left")
    .dropna(subset=["start_time", "ladder", "floor"])
)

# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step filters administrative roles and maps clinical ladder
# levels to non-linear competency weights.
# Below is a simplified baseline assigning a uniform unit weight (1) to all valid staff.
df = df[df["ladder"].isin(LADDER_MAP.keys())].copy()

df["shift_date"] = pd.to_datetime(df["shift_date"])
df["start_datetime"] = pd.to_datetime(df["shift_date"].astype(str) + " " + df["start_time"])
df["end_datetime"] = pd.to_datetime(df["shift_date"].astype(str) + " " + df["end_time"])
df["ladder_converted"] = 1

print(f"Day shift records: {len(df)}")
print(f"Ladder tier distribution (baseline unit weights): {df['ladder_converted'].value_counts().sort_index().to_dict()}")

## 2. Hourly Staffing & Weighted Skill Aggregation

In [ ]:
# --- [REDACTED: PROPRIETARY LOGIC] ---
# Conceptually, this step computes exact fractional hour overlap ratios across shift
# boundaries and multiplies them by non-linear clinical skill weights.
# Below is a simplified uniform unit indicator across the configured shift hours.
ratio_cols = [f"ratio_{h:02d}" for h in HOURS]
ratios_df = pd.DataFrame(1.0, index=df.index, columns=ratio_cols)

staff_hourly = pd.concat(
    [
        df[["floor", "shift_date", "ladder_converted"]].reset_index(drop=True),
        ratios_df.reset_index(drop=True),
    ],
    axis=1,
)

grouped = staff_hourly.groupby(["floor", "shift_date"])
staff_agg = grouped[ratio_cols].sum()
staff_agg.columns = [f"staff_{h:02d}" for h in HOURS]

weighted = staff_hourly[ratio_cols].multiply(staff_hourly["ladder_converted"], axis=0)
ladder_agg = weighted.groupby([staff_hourly["floor"], staff_hourly["shift_date"]]).sum()
ladder_agg.columns = [f"ladder_{h:02d}" for h in HOURS]

hourly_agg = pd.concat([staff_agg, ladder_agg], axis=1).reset_index()
print(f"Hourly aggregation: {hourly_agg.shape}")

## 3. Calculate Floor-Day Summary Metrics

In [ ]:
staff_cols = [f"staff_{h:02d}" for h in HOURS]
ladder_cols = [f"ladder_{h:02d}" for h in HOURS]

result_df = hourly_agg[["floor", "shift_date"]].copy()
result_df["total_staff_hours"] = hourly_agg[staff_cols].sum(axis=1)
result_df["daily_avg_staff_count"] = hourly_agg[staff_cols].mean(axis=1)
result_df["daily_avg_weighted_ladder"] = hourly_agg[ladder_cols].mean(axis=1)
result_df["day_of_week"] = result_df["shift_date"].dt.dayofweek

print(f"Summary: {result_df.shape}")

## 4. Merge Clinical Events, Beacon Features, Busyness, & Care Summary

In [ ]:
date_str = result_df["shift_date"].dt.strftime("%Y-%m-%d")

t_daily_summary = pd.read_csv(PATHS["t_daily_summary"])
t_daily_summary["date"] = pd.to_datetime(t_daily_summary["date"]).dt.strftime("%Y-%m-%d")

t_beacon = pd.read_csv(PATHS["hourly_features"])
t_beacon = t_beacon[t_beacon["shift_id"].isin(day_shift_ids)]
t_beacon = t_beacon[(t_beacon["hour"] >= HOURS[0]) & (t_beacon["hour"] <= HOURS[-1])]
t_beacon["shift_date"] = pd.to_datetime(t_beacon["shift_date"]).dt.strftime("%Y-%m-%d")
beacon_agg = (
    t_beacon.groupby(["floor", "shift_date"])["f_total_time_patient_rooms"]
    .sum()
    .reset_index()
    .rename(columns={"shift_date": "date", "f_total_time_patient_rooms": "total_time_patient_rooms"})
)

t_busyness = pd.read_csv(PATHS["t_daily_busyness"])
t_busyness["date"] = pd.to_datetime(t_busyness["date"]).dt.strftime("%Y-%m-%d")

t_care = pd.read_csv(PATHS["floor_daily_care_level_summary"])
t_care["date"] = pd.to_datetime(t_care["date"]).dt.strftime("%Y-%m-%d")
t_care["floor"] = t_care["ward"].str.extract(r"(\d+)").astype(int)
care_cols = [
    "floor", "date", "care_level_sum",
    "care_mean_per_staff", "care_std_per_staff",
    "care_max_ratio", "care_gini",
]

result_df = (
    result_df.assign(date_str=date_str)
    .merge(t_daily_summary, left_on=["floor", "date_str"], right_on=["floor", "date"], how="inner")
    .drop(columns=["date"])
    .merge(beacon_agg, left_on=["floor", "date_str"], right_on=["floor", "date"], how="inner")
    .drop(columns=["date"])
    .merge(t_busyness, left_on=["floor", "date_str"], right_on=["floor", "date"], how="inner")
    .drop(columns=["date"])
    .merge(t_care[care_cols], left_on=["floor", "date_str"], right_on=["floor", "date"], how="inner")
    .drop(columns=["date", "date_str"])
)

result_df["patient_room_time_per_staff_hour"] = (
    result_df["total_time_patient_rooms"] / result_df["total_staff_hours"]
)

print(f"Final merged shape: {result_df.shape}")

## 5. Save Floor-Daily Staffing Summary

In [ ]:
day_dummies = pd.get_dummies(result_df["day_of_week"], prefix="dow", dtype=int)
for d_i in range(7):
    col_d = f"dow_{d_i}"
    if col_d not in day_dummies.columns:
        day_dummies[col_d] = 0
day_dummies = day_dummies[[f"dow_{i}" for i in range(7)]]

output_df = pd.concat([result_df.drop(columns=["day_of_week"]), day_dummies], axis=1)

output_file = PATHS["floor_daily_staffing_summary"]
output_df.to_csv(output_file, index=False)
print(f"Saved: {output_file.name} ({output_df.shape[0]} rows x {output_df.shape[1]} cols)")